you got this motherfucka

<h1> Demo Implementation

In [2]:
!pip3 install jsonlines

In [50]:
import jsonlines
import os
import torch
from transformers import AutoTokenizer
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
import numpy as np

# Paths to dataset and image folder
DATASET_PATH = "/content/drive/MyDrive/NLP Data/all_data.jsonl"
IMAGE_FOLDER = "/content/drive/MyDrive/NLP Data/img/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize tokenizer (BERT)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

from sklearn.model_selection import train_test_split

# Assuming 'dataset' is your full dataset


# Load dataset
def load_dataset(file_path):
    data = []
    with jsonlines.open(file_path) as reader:
        for obj in reader:
            data.append(obj)
    return data

dataset = load_dataset(DATASET_PATH)
train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)
train_data, val_data = train_test_split(train_data, test_size=0.1, random_state=42)

# Tokenize text data (title, target text, and conversation utterances)
def tokenize_texts(texts, max_length=128):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

# Preprocess dataset entries
def preprocess_dataset(dataset):
    titles, target_texts, conversations, labels, image_paths = [], [], [], [], []

    for entry in dataset:
        # Add title and target text
        titles.append(entry["title"])
        target_texts.append(entry["text"])
        labels.append(entry["bin_label_id"])
        image_paths.append(os.path.join(IMAGE_FOLDER, entry["img"]))

        # Process conversation utterances
        if entry["conv_utterances"]:
            conversations.append(entry["conv_utterances"])
        else:
            conversations.append([""])  # Placeholder for empty conversation

    # Tokenize data
    title_tokens = tokenize_texts(titles)
    text_tokens = tokenize_texts(target_texts)
    conversation_tokens = [tokenize_texts(conv) for conv in conversations]

    return title_tokens, text_tokens, conversation_tokens, torch.tensor(labels), image_paths

train_title_tokens, train_text_tokens, train_conv_tokens, train_labels, train_image_paths = preprocess_dataset(train_data)
val_title_tokens, val_text_tokens, val_conv_tokens, val_labels, val_image_paths = preprocess_dataset(val_data)
test_title_tokens, test_text_tokens, test_conv_tokens, test_labels, test_image_paths = preprocess_dataset(test_data)

In [51]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# Load pre-trained Faster R-CNN model
faster_rcnn = fasterrcnn_resnet50_fpn(pretrained=True).to(DEVICE)
faster_rcnn.eval()

# Image transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Extract features for each image
def extract_image_features(image_paths):
    features = []
    for image_path in tqdm(image_paths):
        image = Image.open(image_path).convert("RGB")
        image = transform(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            output = faster_rcnn.backbone(image)["0"]  # Extract backbone features
            features.append(output.mean(dim=(2, 3)))  # Average pooling
    return torch.stack(features)

image_features = extract_image_features(train_image_paths)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 9109/9109 [14:35<00:00, 10.40it/s]


In [52]:
print("Image features shape:", image_features.shape)

Image features shape: torch.Size([9109, 1, 256])


In [53]:
import torch

# Save image features as a .pt file
torch.save(image_features, "training_image_features.pt")

In [54]:
import numpy as np

# Convert image features to NumPy and save as .npy
np.save("training_image_features.npy", image_features.cpu().numpy())

In [56]:
import torch.nn as nn
from transformers import AutoModel

class MMConvModel(nn.Module):
    def __init__(self, text_hidden_size=768, visual_hidden_size=256):
        super(MMConvModel, self).__init__()
        self.text_hidden_size = text_hidden_size
        self.visual_hidden_size = visual_hidden_size

        # Pre-trained BERT for text encoding
        self.text_encoder = AutoModel.from_pretrained("bert-base-uncased")

        # Projection layer for visual features
        self.visual_proj = nn.Linear(visual_hidden_size, text_hidden_size)

        # Fusion layer for title and visual
        self.title_visual_fusion = nn.Linear(2 * text_hidden_size, text_hidden_size)

        # TransformerEncoder for fusion
        self.fusion = nn.TransformerEncoderLayer(d_model=text_hidden_size, nhead=8)

        # Final projection and classification layers
        self.proj = nn.Linear(2 * text_hidden_size, text_hidden_size)
        self.cls = nn.Linear(text_hidden_size, 1)

        # Activation and dropout
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, title, visual, conversation, text):
        # Encode title
        title_hidden = self.text_encoder(
            input_ids=title["input_ids"],
            attention_mask=title["attention_mask"]
        ).last_hidden_state[:, 0, :]  # [CLS]

        # Project visual features to text_hidden_size
        visual_hidden = self.visual_proj(visual)

        # Fuse title and visual
        fused_title_visual = torch.cat((title_hidden, visual_hidden), dim=1)
        fused_title_visual = self.title_visual_fusion(fused_title_visual)
        fused_title_visual = self.fusion(fused_title_visual.unsqueeze(0)).squeeze(0)

        # Encode target text
        text_hidden = self.text_encoder(
            input_ids=text["input_ids"],
            attention_mask=text["attention_mask"]
        ).last_hidden_state[:, 0, :]

        # Final fusion: Concatenate fused title-visual and text hidden states
        final_hidden = torch.cat((fused_title_visual, text_hidden), dim=1)

        # Project back to text_hidden_size
        final_hidden = self.proj(final_hidden)

        # Apply TransformerEncoder for fusion
        fused_output = self.fusion(final_hidden.unsqueeze(0)).squeeze(0)

        # Classification
        logits = self.cls(self.dropout(self.gelu(fused_output)))
        return logits

In [57]:
from torch.utils.data import DataLoader, TensorDataset

# Prepare dataset for DataLoader
# dataset = TensorDataset(title_tokens["input_ids"], text_tokens["input_ids"], image_features, labels)
# data_loader = DataLoader(dataset, batch_size=16, shuffle=True)

# Prepare dataset for DataLoader
dataset = TensorDataset(
    train_title_tokens["input_ids"],
    train_title_tokens["attention_mask"],
    train_text_tokens["input_ids"],
    train_text_tokens["attention_mask"],
    image_features,
    train_labels
)

data_loader = DataLoader(dataset, batch_size=16, shuffle=True)


# Initialize model
model = MMConvModel(text_hidden_size=768, visual_hidden_size=256).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)
criterion = nn.BCEWithLogitsLoss()

# Training loop
# Training loop
for epoch in range(10):
    model.train()
    epoch_loss = 0
    for batch in data_loader:
        title_ids, title_mask, text_ids, text_mask, visual, label = [b.to(DEVICE) for b in batch]

        # Create input dictionaries for BERT
        title = {"input_ids": title_ids, "attention_mask": title_mask}
        text = {"input_ids": text_ids, "attention_mask": text_mask}

        # Ensure visual has the correct shape [batch_size, 256]
        visual = visual.squeeze(1)

        # Forward pass
        logits = model(title, visual, [], text)  # Replace [] with conversation input when available
        loss = criterion(logits.view(-1), label.float())

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {epoch_loss / len(data_loader)}")

Epoch 1, Loss: 0.5584355562925338
Epoch 2, Loss: 0.4157431906906136
Epoch 3, Loss: 0.23157962534511298
Epoch 4, Loss: 0.10955141975969207
Epoch 5, Loss: 0.06877585821090626
Epoch 6, Loss: 0.053220443315027906
Epoch 7, Loss: 0.03870277668802852
Epoch 8, Loss: 0.0344098224258663
Epoch 9, Loss: 0.026323931113195916
Epoch 10, Loss: 0.034149994854937846


In [58]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_metrics(y_true, y_pred_probs):
    y_pred = (y_pred_probs >= 0.5).astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred_probs)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

def evaluate_model(model, data_loader):
    model.eval()
    all_labels = []
    all_logits = []

    with torch.no_grad():
        for batch in data_loader:
            title_ids, title_mask, text_ids, text_mask, visual, label = [b.to(DEVICE) for b in batch]

            # Create input dictionaries for BERT
            title = {"input_ids": title_ids, "attention_mask": title_mask}
            text = {"input_ids": text_ids, "attention_mask": text_mask}

            # Ensure visual has the correct shape [batch_size, 256]
            visual = visual.squeeze(1)

            # Forward pass
            logits = model(title, visual, [], text)  # Replace [] with conversation input when available
            probs = torch.sigmoid(logits).view(-1).cpu().numpy()

            # Store predictions and true labels
            all_logits.extend(probs)
            all_labels.extend(label.cpu().numpy())

    # Compute metrics
    metrics = compute_metrics(np.array(all_labels), np.array(all_logits))
    return metrics

image_features = extract_image_features(val_image_paths)

# Create validation/test dataset
val_dataset = TensorDataset(
    val_title_tokens["input_ids"],
    val_title_tokens["attention_mask"],
    val_text_tokens["input_ids"],
    val_text_tokens["attention_mask"],
    image_features,
    val_labels
)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Evaluate the model
val_metrics = evaluate_model(model, val_loader)

# Print evaluation metrics
print("Validation Metrics:")
for metric, value in val_metrics.items():
    print(f"{metric}: {value:.4f}")

100%|██████████| 1013/1013 [01:38<00:00, 10.24it/s]


Validation Metrics:
accuracy: 0.7058
precision: 0.5941
recall: 0.7690
f1: 0.6704
auc: 0.7759


In [41]:
import torch

# Save the model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epoch,
    'loss': loss
}, 'mmconv_model_train.pth')

2. Load the Saved Features
In future runs, you can directly load the saved features instead of recalculating them.

Example Code for Loading:

python
Copy code
# Load image features from .pt file
image_features = torch.load("image_features.pt").to(DEVICE)
Or, if you saved the features as a NumPy array:

python
Copy code
# Load image features from .npy file
image_features = torch.tensor(np.load("image_features.npy")).to(DEVICE)


In [59]:
image_features = extract_image_features(test_image_paths)

test_dataset = TensorDataset(
    test_title_tokens["input_ids"],  # Adjust indices as needed
    test_title_tokens["attention_mask"],
    test_text_tokens["input_ids"],
    test_text_tokens["attention_mask"],
    image_features,
    test_labels
)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Evaluate the model on test set
test_metrics = evaluate_model(model, test_loader)

print("Test Metrics:")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

100%|██████████| 2531/2531 [03:55<00:00, 10.76it/s]


Test Metrics:
accuracy: 0.6934
precision: 0.6173
recall: 0.7734
f1: 0.6866
auc: 0.7675


In [61]:
def evaluate_model_with_examples(model, data_loader, tokenizer):
    model.eval()
    all_labels = []
    all_logits = []
    examples = []

    with torch.no_grad():
        for batch in data_loader:
            title_ids, title_mask, text_ids, text_mask, visual, label = [b.to(DEVICE) for b in batch]

            title = {"input_ids": title_ids, "attention_mask": title_mask}
            text = {"input_ids": text_ids, "attention_mask": text_mask}

            visual = visual.squeeze(1)

            logits = model(title, visual, [], text)
            probs = torch.sigmoid(logits).view(-1).cpu().numpy()

            all_logits.extend(probs)
            all_labels.extend(label.cpu().numpy())

            for i in range(len(label)):
                examples.append({
                    "title": tokenizer.decode(title_ids[i]),
                    "text": tokenizer.decode(text_ids[i]),
                    "true_label": label[i].item(),
                    "predicted_prob": probs[i],
                    "predicted_label": 1 if probs[i] >= 0.5 else 0
                })

    metrics = compute_metrics(np.array(all_labels), np.array(all_logits))
    return metrics, examples

# Evaluate the model on test set
test_metrics, test_examples = evaluate_model_with_examples(model, test_loader, tokenizer)

print("Test Metrics:")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

# Print details of some examples
for i, example in enumerate(test_examples[:50]):  # Print first 5 examples
    print(f"\nExample {i+1}:")
    print(f"Title: {example['title']}")
    print(f"Text: {example['text'][:100]}...")  # Print first 100 characters of text
    print(f"True Label: {'Derailed' if example['true_label'] == 1 else 'Not Derailed'}")
    print(f"Predicted Label: {'Derailed' if example['predicted_label'] == 1 else 'Not Derailed'}")
    print(f"Predicted Probability: {example['predicted_prob']:.4f}")

Test Metrics:
accuracy: 0.6934
precision: 0.6173
recall: 0.7734
f1: 0.6866
auc: 0.7675

Example 1:
Title: [CLS] this sign outside of mcdonald ' s [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
Text: [CLS] yeah, we kind of tried to talk about it a few times and now they will use it forever against u...
True Label: Derailed
Pre